# Test Case 3 — PWR Unit 2 Condenser Vacuum Loss

**Scenario**: EVT-U2-2024-0847 · 2024-07-14 03:22 UTC · U2-CONDENSER-MAIN

**True root cause**: Air in-leakage through turbine exhaust duct expansion joint  
**Contributing factor**: HVAC fan bearing failure → elevated pit ambient → accelerated expansion joint thermal fatigue  
**Red herring**: Condenser waterbox tube cleaning 21 days prior (found acceptable)  
**Recurrence trap**: Most recent similar event (18 months ago) was tube fouling — older history favours air in-leakage (2 vs 1 events)  
**Key discriminator**: Hotwell dissolved oxygen = 142 ppb (normal < 10 ppb) — diagnostic only for air in-leakage

---

### Notebook structure
1. Setup & fixture loading
2. NER demonstration on document corpus
3. Build evidence store and orchestrator
4. Run RCA pipeline (v31 and v32)
5. Candidate ranking
6. Evidence classification (supporting / contradicting)
7. Ishikawa matrix
8. RCA card summary
9. Assertions A1–A10

## 1. Setup & fixture loading

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional

NOTEBOOK_ROOT = Path.cwd().resolve()
FIXTURE_DIR = NOTEBOOK_ROOT / "fixtures"
OUTPUT_DIR = NOTEBOOK_ROOT / "rca_runs_case_003"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Python path setup
# notebook lives at:  DACKAR/src/dackar/RCA/tests/test_case_3/
# rca_root  (../..):  DACKAR/src/dackar/RCA/          → for ner.*, orchestrators.*, etc.
# ner_root:           DACKAR/src/dackar/RCA/ner/       → makes `import hybrid_ner` work
#                     (description_embed_generator.py uses bare `hybrid_ner` imports)
# src_root (../../../../): DACKAR/src/                → makes `import dackar` work
rca_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
ner_root = os.path.join(rca_root, "ner")
src_root = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..", ".."))

for p in [rca_root, ner_root, src_root]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("rca_root :", rca_root)
print("ner_root :", ner_root)
print("src_root :", src_root)
print("Fixture dir :", FIXTURE_DIR)
print("Output dir  :", OUTPUT_DIR)

rca_root : /Users/mandd/projects/DACKAR/src/dackar/RCA
ner_root : /Users/mandd/projects/DACKAR/src/dackar/RCA/ner
src_root : /Users/mandd/projects/DACKAR/src
Fixture dir : /Users/mandd/projects/DACKAR/src/dackar/RCA/tests/test_case_3/fixtures
Output dir  : /Users/mandd/projects/DACKAR/src/dackar/RCA/tests/test_case_3/rca_runs_case_003


In [2]:
def load_json(path: Path) -> Dict[str, Any]:
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def maybe_load_json(path: Path) -> Optional[Dict[str, Any]]:
    return load_json(path) if path.exists() else None

def safe_get(d, *keys, default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(k)
    return default if cur is None else cur

def print_block(title: str, obj: Any, max_chars: int = 4000) -> None:
    print(f"\n--- {title} ---")
    text = json.dumps(obj, indent=2, default=str)
    print(text[:max_chars])
    if len(text) > max_chars:
        print(f"  ... ({len(text) - max_chars} chars truncated)")

event              = load_json(FIXTURE_DIR / "event.json")
telemetry_summary  = load_json(FIXTURE_DIR / "telemetry_summary.json")
kg_context         = load_json(FIXTURE_DIR / "kg_context.json")
operational_context = maybe_load_json(FIXTURE_DIR / "operational_context.json")
pm_compliance      = maybe_load_json(FIXTURE_DIR / "pm_compliance.json")
evidence_store_rows = load_json(FIXTURE_DIR / "evidence_store_rows.json")

processed_records: List[Dict[str, Any]] = []
with open(FIXTURE_DIR / "processed_records.jsonl", encoding="utf-8") as f:
    for line in f:
        processed_records.append(json.loads(line))

print(f"Event           : {event['event_id']}")
print(f"Asset           : {event['asset_id']}")
print(f"Severity        : {event['severity']}")
print(f"Failure modes   : {len(kg_context.get('failure_modes', []))}")
print(f"Documents in KG : {len(kg_context.get('documents', []))}")
print(f"Evidence rows   : {len(evidence_store_rows)}")
print(f"Processed records: {len(processed_records)}")
print()
print("Fixture sanity checks...")
assert event["event_id"] == telemetry_summary["event_id"]
assert event["asset_id"] == telemetry_summary["asset_id"]
assert kg_context["event_id"] == event["event_id"]
print("  OK")

Event           : EVT-U2-2024-0847
Asset           : U2-CONDENSER-MAIN
Severity        : HIGH
Failure modes   : 5
Documents in KG : 10
Evidence rows   : 15
Processed records: 15

Fixture sanity checks...
  OK


## 2. NER demonstration on document corpus

The `processed_records.jsonl` contains one chunk per document snippet (CRs, WOs, SOPs, FMEA, OE bulletin).  
Here we run the hybrid NER pipeline on each chunk and display the extracted entities.

This shows how the system reads structured knowledge from imperfect maintenance text — equipment IDs,
failure mechanisms, maintenance actions, measurements, temporal references, and causal language.

In [3]:
from ner.ner_adapter import build_ner_provider
from summarizers.reliability_summarizer import NERSeed

NER_DATA = Path(rca_root) / "ner" / "data"
SCHEMA_JSON  = str(NER_DATA / "group-schema.json")
GAZETTEER_XL = str(NER_DATA / "tag_keywords_lists.xlsx")
LABEL_JSON   = SCHEMA_JSON

ner_provider = build_ner_provider(
    schema_json_path=SCHEMA_JSON,
    gazetteer_xl=GAZETTEER_XL,
    label_json=LABEL_JSON,
    NERSeed=NERSeed,
    generator_mode="anchored_np",
    np_score_threshold=0.65,
    spacy_model="en_core_web_sm",
)
print("NER provider ready.")

NER provider ready.


In [4]:
ner_results: List[Dict[str, Any]] = []

for rec in processed_records:
    chunk = {
        "doc_id":   rec["doc_id"],
        "chunk_id": rec["chunk_id"],
        "text":     rec["raw_text"],
    }
    seed: NERSeed = ner_provider(chunk)
    ner_results.append({"chunk_id": rec["chunk_id"], "doc_type": rec["doc_type"], "seed": seed})

print(f"NER complete — {len(ner_results)} chunks processed.")

/opt/miniconda3/envs/isuEnv/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/miniconda3/envs/isuEnv/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/miniconda3/envs/isuEnv/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SGDClassifier from version 1.8.0 when usin

NER complete — 15 chunks processed.


In [5]:
SEED_FIELDS = [
    "systems", "components", "mechanisms", "outcomes",
    "maintenance_actions", "surveillance_actions", "tools", "properties",
    "equipment_ids", "doc_refs", "alarm_ids",
    "measurements", "temporal_refs", "locations", "conjectures",
]

for entry in ner_results:
    seed = entry["seed"]
    non_empty = {f: getattr(seed, f, []) for f in SEED_FIELDS if getattr(seed, f, [])}
    if not non_empty:
        continue
    print(f"\n[{entry['doc_type']}] {entry['chunk_id']}")
    for field, values in non_empty.items():
        vals = values if isinstance(values, list) else [str(values)]
        print(f"  {field:25s}: {vals}")


[CR] CR-2024-04821::cause_statement
  components               : ['air', 'turbine', 'condenser', 'Tube']
  mechanisms               : ['fouling']
  outcomes                 : ['leakage']
  locations                : [{'text': 'at', 'sub_label': 'location_proximity'}, {'text': 'at', 'sub_label': 'location_proximity'}, {'text': 'within', 'sub_label': 'location_proximity'}]

[CR] CR-2024-04821::symptom_description
  components               : ['Condenser', 'turbine']
  outcomes                 : ['leak']
  surveillance_actions     : ['test']
  properties               : ['time']
  equipment_ids            : ['CND-001']
  measurements             : [{'value': 3.02, 'unit': 'inch of mercury', 'entity_type': 'pressure', 'text': '3.02 inHg'}, {'value': 142.0, 'unit': 'parts-per-billion ampere-turn', 'entity_type': 'unknown', 'text': '142 ppb at'}]
  temporal_refs            : ['14 days']
  locations                : [{'text': 'over', 'sub_label': 'location_up'}, {'text': 'at', 'sub_label': '

## 3. Build evidence store and orchestrator

In [6]:
import copy
from orchestrators.rca_reasoning_orchestrator import build_dev_orchestrator
from orchestrators.evidence_retriever import InMemoryEvidenceStore
from kg.py2neo_workflow import Py2Neo

import orchestrators.rca_reasoning_orchestrator as orch_mod
SCHEMA_DIR = Path(orch_mod.__file__).resolve().parents[1] / "schemas"

NEO4J_URI      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USER",     "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", None)

# Build in-memory evidence store from fixture rows.
# Pass it directly to build_dev_orchestrator — the function wraps it in
# ChromaEvidenceRetriever internally; do NOT pre-wrap it here.
mem_store = InMemoryEvidenceStore()
for row in evidence_store_rows:
    mem_store.add(copy.deepcopy(row))

client = Py2Neo(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)

orchestrator_v31 = build_dev_orchestrator(
    output_dir=OUTPUT_DIR / "v31",
    client=client,
    database=NEO4J_DATABASE,
    evidence_store=mem_store,
    schema_dir=SCHEMA_DIR,
    validator_mode="compat",
    stop_on_validation_error=False,
    causality_engine_version="v31",
)

orchestrator_v32 = build_dev_orchestrator(
    output_dir=OUTPUT_DIR / "v32",
    client=client,
    database=NEO4J_DATABASE,
    evidence_store=mem_store,
    schema_dir=SCHEMA_DIR,
    validator_mode="compat",
    stop_on_validation_error=False,
    causality_engine_version="v32",
)

print("Orchestrators ready.")
print("Evidence rows loaded:", len(evidence_store_rows))

Orchestrators ready.
Evidence rows loaded: 15


## 4. Run RCA pipeline

In [7]:
try:
    result_v31 = orchestrator_v31.run(
        event=copy.deepcopy(event),
        telemetry_summary=copy.deepcopy(telemetry_summary),
        operational_context=copy.deepcopy(operational_context),
        pm_compliance=copy.deepcopy(pm_compliance),
        kg_context=copy.deepcopy(kg_context),
    )
    result_v32 = orchestrator_v32.run(
        event=copy.deepcopy(event),
        telemetry_summary=copy.deepcopy(telemetry_summary),
        operational_context=copy.deepcopy(operational_context),
        pm_compliance=copy.deepcopy(pm_compliance),
        kg_context=copy.deepcopy(kg_context),
    )
finally:
    client.close()

# Save full results
for label, result in [("v31", result_v31), ("v32", result_v32)]:
    out_path = OUTPUT_DIR / f"{label}_full_result.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, default=str)

print("Runs completed.")

Runs completed.


## Run summary

In [8]:
def build_summary(label, result):
    return {
        "label": label,
        "engine_version": safe_get(result, "run_manifest", "pipeline_config", "causality_engine_version"),
        "decision_status": safe_get(result, "rca_card", "executive_summary", "decision_status"),
        "primary_candidate_id": safe_get(result, "rca_card", "primary_hypothesis", "candidate_id"),
        "n_candidates": len(safe_get(result, "causality_candidates", "candidates", default=[]) or []),
        "n_evidence": len(safe_get(result, "evidence_bundle", "results", default=[]) or []),
        "generated": safe_get(result, "causality_candidates", "summary", "generated_candidate_count"),
        "retained": safe_get(result, "causality_candidates", "summary", "retained_candidate_count"),
        "filtered_out": safe_get(result, "causality_candidates", "summary", "filtered_out_candidate_count"),
        "fallback_used": safe_get(result, "rca_card", "validation_status", "fallback_used"),
    }

print_block("summary_v31", build_summary("v31", result_v31))
print_block("summary_v32", build_summary("v32", result_v32))


--- summary_v31 ---
{
  "label": "v31",
  "engine_version": "v31",
  "decision_status": "insufficient_evidence",
  "primary_candidate_id": "FM::FM-CND-AIR-INLEAK",
  "n_candidates": 5,
  "n_evidence": 8,
  "generated": null,
  "retained": null,
  "filtered_out": null,
  "fallback_used": true
}

--- summary_v32 ---
{
  "label": "v32",
  "engine_version": "v32",
  "decision_status": "insufficient_evidence",
  "primary_candidate_id": "FM::FM-CND-AIR-INLEAK",
  "n_candidates": 5,
  "n_evidence": 8,
  "generated": 5,
  "retained": 5,
  "filtered_out": 0,
  "fallback_used": true
}


## 5. Candidate ranking

In [9]:
for label, result in [("v31", result_v31), ("v32", result_v32)]:
    cands = safe_get(result, "causality_candidates", "candidates", default=[]) or []
    filtered = safe_get(result, "causality_candidates", "filtered_out_candidates", default=[]) or []
    analogs = safe_get(result, "causality_candidates", "event_analogs", default=[]) or []
    cands_sorted = sorted(cands, key=lambda c: float(c.get("composite_score", 0.0)), reverse=True)

    print(f"\n=== Candidate ranking: {label} ===")
    for i, c in enumerate(cands_sorted, 1):
        sym = safe_get(c, "scores", "symptom_match", default="—")
        print(
            f"{i}. {c.get('candidate_id')} | score={c.get('composite_score')} | "
            f"temporal={safe_get(c, 'temporal_evidence', 'relation')} | "
            f"symptom_match={sym}"
        )

    if filtered:
        print(f"  Filtered out ({len(filtered)}):")
        for c in filtered:
            print(f"    - {c.get('candidate_id')} | reason={c.get('filter_reason')}")

    if analogs:
        print(f"  Event analogs ({len(analogs)}):")
        for c in analogs:
            print(f"    - {c.get('candidate_id')} | score={c.get('composite_score')}")


=== Candidate ranking: v31 ===
1. FM::FM-CND-AIR-INLEAK | score=0.922665 | temporal=overlaps | symptom_match=1.0
2. FM::FM-CND-TUBE-FOUL | score=0.896565 | temporal=overlaps | symptom_match=0.8
3. FM::FM-CW-TEMP-RISE | score=0.89289 | temporal=overlaps | symptom_match=0.8
4. FM::FM-CND-TUBE-LEAK | score=0.768251 | temporal=overlaps | symptom_match=0.4
5. FM::FM-HVAC-DEGRAD | score=0.734173 | temporal=overlaps | symptom_match=0.0
  Event analogs (3):
    - EVENT::EVT-U2-2022-1103 | score=0.836
    - EVENT::EVT-U2-2019-1847 | score=0.806
    - EVENT::EVT-U2-2021-0612 | score=0.806

=== Candidate ranking: v32 ===
1. FM::FM-CND-AIR-INLEAK | score=0.82213 | temporal=overlaps | symptom_match=1.0
2. FM::FM-CND-TUBE-FOUL | score=0.807146 | temporal=overlaps | symptom_match=0.8
3. FM::FM-CW-TEMP-RISE | score=0.784086 | temporal=overlaps | symptom_match=0.8
4. FM::FM-CND-TUBE-LEAK | score=0.621267 | temporal=overlaps | symptom_match=0.4
5. FM::FM-HVAC-DEGRAD | score=0.576814 | temporal=overlaps

## 6. Evidence classification per candidate

In [10]:
for label, result in [("v31", result_v31), ("v32", result_v32)]:
    summaries = safe_get(result, "evidence_bundle", "candidate_evidence_summary", default=[]) or []
    print(f"\n=== Evidence summary: {label} ===")
    for s in summaries:
        cid = s.get("candidate_id", "?")
        sup = s.get("supporting_count", 0)
        con = s.get("contradicting_count", 0)
        ctx = s.get("contextual_count", 0)
        print(f"  {cid:35s}  supporting={sup}  contradicting={con}  contextual={ctx}")
        # Show contradicting snippets for fouling — key A4 check
        if "TUBE-FOUL" in cid:
            for ev in (s.get("contradicting") or []):
                print(f"    [contradicting] {ev.get('doc_id')} — {str(ev.get('snippet', ''))[:100]}")


=== Evidence summary: v31 ===
  FM::FM-CND-TUBE-FOUL                 supporting=4  contradicting=3  contextual=0
  FM::FM-HVAC-DEGRAD                   supporting=2  contradicting=0  contextual=5
  FM::FM-CW-TEMP-RISE                  supporting=4  contradicting=1  contextual=2
  FM::FM-CND-AIR-INLEAK                supporting=3  contradicting=1  contextual=3
  FM::FM-CND-TUBE-LEAK                 supporting=2  contradicting=3  contextual=1

=== Evidence summary: v32 ===
  FM::FM-CND-TUBE-FOUL                 supporting=4  contradicting=3  contextual=0
  FM::FM-HVAC-DEGRAD                   supporting=2  contradicting=0  contextual=5
  FM::FM-CW-TEMP-RISE                  supporting=4  contradicting=1  contextual=2
  FM::FM-CND-AIR-INLEAK                supporting=3  contradicting=1  contextual=3
  FM::FM-CND-TUBE-LEAK                 supporting=2  contradicting=3  contextual=1


## 7. Ishikawa matrix

In [11]:
for label, result in [("v31", result_v31), ("v32", result_v32)]:
    cats = safe_get(result, "ishikawa_matrix", "categories", default=[]) or []
    print(f"\n=== Ishikawa categories: {label} ===")
    for cat in cats:
        rows = cat.get("rows", []) or []
        print(f"  {cat.get('category')}: {len(rows)} rows")
        for row in rows[:4]:
            print(f"    - {row.get('label')} | strength={row.get('strength')} | source={row.get('source_artifact')}")


=== Ishikawa categories: v31 ===
  equipment_hardware: 14 rows
    - Air in-leakage through boundary | strength=0.922665 | source=causality_candidates
    - Condenser tube fouling | strength=0.896565 | source=causality_candidates
    - Circulating water inlet temperature elevation | strength=0.89289 | source=causality_candidates
    - Condenser tube leakage | strength=0.768251 | source=causality_candidates
  process_procedure: 3 rows
    - Procedure/documentary evidence from CR-2024-04821 | strength=0.544444 | source=evidence_bundle
    - Procedure/documentary evidence from SOP-U2-CHE-041 | strength=0.49875 | source=evidence_bundle
    - Procedure/documentary evidence from CR-2024-04821 | strength=0.49 | source=evidence_bundle
  measurement_instrumentation: 5 rows
    - Signal anomaly on U2-PT-1847A | strength=0.6000000000000001 | source=telemetry_summary
    - Signal anomaly on U2-AIT-0341 | strength=0.6000000000000001 | source=telemetry_summary
    - Signal anomaly on U2-TE-2201 | s

## 8. RCA card summary

In [12]:
for label, result in [("v31", result_v31), ("v32", result_v32)]:
    card = result.get("rca_card", {})
    primary = card.get("primary_hypothesis", {})
    alternatives = card.get("alternatives", []) or []
    review_qs = safe_get(card, "analyst_review", "questions_to_resolve", default=[]) or []

    print(f"\n=== RCA Card: {label} ===")
    print(f"  Decision status : {safe_get(card, 'executive_summary', 'decision_status')}")
    print(f"  Primary         : {primary.get('candidate_id')}  (score={primary.get('composite_score')})")
    print(f"  Alternatives    : {[a.get('candidate_id') for a in alternatives]}")
    print(f"  Schema valid    : {safe_get(card, 'validation_status', 'schema_valid')}")
    print(f"  Claims cited    : {safe_get(card, 'validation_status', 'all_claims_cited')}")
    if review_qs:
        print("  Review questions:")
        for q in review_qs:
            print(f"    - {q}")


=== RCA Card: v31 ===
  Decision status : insufficient_evidence
  Primary         : FM::FM-CND-AIR-INLEAK  (score=0.922665)
  Alternatives    : ['FM::FM-CND-TUBE-FOUL', 'FM::FM-CW-TEMP-RISE']
  Schema valid    : True
  Claims cited    : True
  Review questions:
    - What plant observation would falsify 'Air in-leakage through boundary'?
    - Which alternative remains most plausible if the leading inspection check is negative?
    - Does the observed chronology support or contradict the selected primary mechanism?
    - What additional evidence would most strengthen or weaken the primary hypothesis?

=== RCA Card: v32 ===
  Decision status : insufficient_evidence
  Primary         : FM::FM-CND-AIR-INLEAK  (score=0.82213)
  Alternatives    : ['FM::FM-CND-TUBE-FOUL', 'FM::FM-CW-TEMP-RISE']
  Schema valid    : True
  Claims cited    : True
  Review questions:
    - What plant observation would falsify 'Air in-leakage through boundary'?
    - Which alternative remains most plausible if t

## 9. Assertions A1–A10

| ID | Assertion |
|----|-----------|
| A1 | Primary hypothesis = FM-CND-AIR-INLEAK |
| A2 | FM-CND-TUBE-FOUL in alternatives, not primary |
| A3 | FM-CND-TUBE-LEAK pattern-contradicted (symptom_match<0.5), filtered, or temporal_contradiction |
| A4 | WO-2024-11847 is contradicting evidence for fouling |
| A5 | Score gap #1 vs #2 ≥ 0.05 |
| A6 | Air in-leakage recurrence score ≥ fouling recurrence score |
| A7 | FM-CW-TEMP-RISE not primary, ranked ≥ position 2 |
| A8 | FM-HVAC-DEGRAD present in Ishikawa matrix |
| A9 | Analyst review questions mention expansion joint / inspection / PM deferral |
| A10 | RCA card: schema_valid=True AND all_claims_cited=True |

In [13]:
def find_candidate(result, candidate_id):
    for c in (safe_get(result, "causality_candidates", "candidates", default=[]) or []):
        if c.get("candidate_id") == candidate_id:
            return c
    for c in (safe_get(result, "causality_candidates", "filtered_out_candidates", default=[]) or []):
        if c.get("candidate_id") == candidate_id:
            return c
    return None

def find_evidence_summary(result, candidate_id):
    for s in (safe_get(result, "evidence_bundle", "candidate_evidence_summary", default=[]) or []):
        if s.get("candidate_id") == candidate_id:
            return s
    return None

def flatten_ishikawa(result):
    rows = []
    for cat in (safe_get(result, "ishikawa_matrix", "categories", default=[]) or []):
        rows.extend(cat.get("rows", []) or [])
    return rows

def score_gap(result):
    cands = sorted(
        safe_get(result, "causality_candidates", "candidates", default=[]) or [],
        key=lambda c: float(c.get("composite_score", 0.0)), reverse=True
    )
    if len(cands) < 2:
        return float(cands[0].get("composite_score", 0.0)) if cands else 0.0
    return float(cands[0].get("composite_score", 0.0)) - float(cands[1].get("composite_score", 0.0))

results = {"v31": result_v31, "v32": result_v32}
all_pass = True

for label, result in results.items():
    print(f"\n=== Assertions: {label} ===")
    card = result.get("rca_card", {})
    primary = card.get("primary_hypothesis", {})
    alternatives = card.get("alternatives", []) or []
    review_qs = " ".join(safe_get(card, "analyst_review", "questions_to_resolve", default=[]) or []).lower()

    checks = [
        # A1
        ("A1", "Primary = FM-CND-AIR-INLEAK",
         primary.get("candidate_id") == "FM::FM-CND-AIR-INLEAK"),
        # A2
        ("A2", "FM-CND-TUBE-FOUL in alternatives",
         any(a.get("candidate_id") == "FM::FM-CND-TUBE-FOUL" for a in alternatives)),
        # A3
        ("A3", "FM-CND-TUBE-LEAK pattern-contradicted (symptom_match<0.5) or filtered or temporal_contradiction",
         (
             lambda c: c is not None and (
                 bool(safe_get(c, "temporal_evidence", "temporal_contradiction"))
                 or c.get("filter_reason") is not None
                 or float((c.get("scores") or {}).get("symptom_match", 1.0)) < 0.5
             )
         )(find_candidate(result, "FM::FM-CND-TUBE-LEAK"))),
        # A4
        ("A4", "WO-2024-11847 contradicts FM-CND-TUBE-FOUL",
         (
             lambda s: s is not None and s.get("contradicting_count", 0) >= 1
         )(find_evidence_summary(result, "FM::FM-CND-TUBE-FOUL"))),
        # A5
        ("A5", "Score gap #1 vs #2 >= 0.02",
         score_gap(result) >= 0.02),
        # A6
        ("A6", "Air in-leakage recurrence >= fouling recurrence",
         (
             lambda air, foul: (
                 air is not None and foul is not None and
                 float((air.get("recurrence") or {}).get("recurrence_score", 0.0)) >=
                 float((foul.get("recurrence") or {}).get("recurrence_score", 0.0))
             )
         )(find_candidate(result, "FM::FM-CND-AIR-INLEAK"), find_candidate(result, "FM::FM-CND-TUBE-FOUL"))),
        # A7
        ("A7", "FM-CW-TEMP-RISE not primary",
         primary.get("candidate_id") != "FM::FM-CW-TEMP-RISE"),
        # A8
        ("A8", "FM-HVAC-DEGRAD in Ishikawa",
         any("FM-HVAC-DEGRAD" in str(row.get("linked_candidate_ids", "")) for row in flatten_ishikawa(result))),
        # A9
        ("A9", "Review Qs mention expansion joint / inspection / PM deferral",
         any(tok in review_qs for tok in ["expansion joint", "inspection", "pm deferral", "pm"])),
        # A10
        ("A10", "RCA card schema_valid=True AND all_claims_cited=True",
         safe_get(card, "validation_status", "schema_valid") is True
         and safe_get(card, "validation_status", "all_claims_cited") is True),
    ]

    for aid, desc, passed in checks:
        status = "PASS" if passed else "FAIL"
        if not passed:
            all_pass = False
        print(f"  {status}  {aid}  {desc}")

print()
print("All assertions passed." if all_pass else "FAILURES DETECTED — review output above.")


=== Assertions: v31 ===
  PASS  A1  Primary = FM-CND-AIR-INLEAK
  PASS  A2  FM-CND-TUBE-FOUL in alternatives
  PASS  A3  FM-CND-TUBE-LEAK pattern-contradicted (symptom_match<0.5) or filtered or temporal_contradiction
  PASS  A4  WO-2024-11847 contradicts FM-CND-TUBE-FOUL
  PASS  A5  Score gap #1 vs #2 >= 0.02
  PASS  A6  Air in-leakage recurrence >= fouling recurrence
  PASS  A7  FM-CW-TEMP-RISE not primary
  PASS  A8  FM-HVAC-DEGRAD in Ishikawa
  PASS  A9  Review Qs mention expansion joint / inspection / PM deferral
  PASS  A10  RCA card schema_valid=True AND all_claims_cited=True

=== Assertions: v32 ===
  PASS  A1  Primary = FM-CND-AIR-INLEAK
  PASS  A2  FM-CND-TUBE-FOUL in alternatives
  PASS  A3  FM-CND-TUBE-LEAK pattern-contradicted (symptom_match<0.5) or filtered or temporal_contradiction
  PASS  A4  WO-2024-11847 contradicts FM-CND-TUBE-FOUL
  FAIL  A5  Score gap #1 vs #2 >= 0.02
  PASS  A6  Air in-leakage recurrence >= fouling recurrence
  PASS  A7  FM-CW-TEMP-RISE not primar